# NB_00_ORCHES_SLV_TO_GLD — Điều phối luồng tính lại silver → silver L2 → gold (dùng chung mọi luồng)

(GOLD_POI_FLOW_DESIGN): DAG dựng từ
`ctrl_mng_pipeline_config` theo `p_pl_name` (cạnh `load_mode = 'RECOMPUTE'`). Thêm luồng mới = thêm dòng config +
notebook node (`NB_<UPPER(trg_tbl)>` hoặc cột `nb_name`) + 1 pipeline gọi notebook này với `p_pl_name` mới.
Luồng nhanh / chậm dùng chung node: chuyển `pl_name` của cạnh (§11), không sửa code.

| Tham số | Mặc định | Ý nghĩa |
|---|---|---|
| `p_pl_name` | (bắt buộc) | pl cần chạy, vd `PL_VV_TRANSFORM_SLV_TO_GLD_1H`; nhịp đọc từ hậu tố `_15M` / `_1H` / `_1D` |
| `p_run_id` | `""` | RunId pipeline (`@pipeline().RunId`); trống = exec_id |
| `p_mode` | `RUN` | `RUN` ; `PLAN` = chỉ in kế hoạch + DAG, không khoá, không ghi gì ; `DRY_RUN` = chạy node ở chế độ không ghi bảng đích, không tiến cạnh |
| `p_max_parallel` | `3` | Số node chạy cùng lúc (runMultiple `concurrency`) |
| `p_node_timeout_min` | `20` | Timeout 1 node (`timeoutPerCellInSeconds`: cell `run_node` là cell dài nhất) |
| `p_dag_timeout_min` | `45` | Timeout cả DAG (`timeoutInSeconds`) |
| `p_lock_timeout_min` | `90` | Khoá luồng quá hạn sau từng này phút; phải > `p_dag_timeout_min` + 5. Timeout activity pipeline phải nhỏ hơn (bài học H1) |
| `p_force_nodes` | `""` | Danh sách `trg_tbl` ép chạy (bỏ qua hộp thư), vd sau khi tạo lại bảng đầu vào hoặc sửa tay bảng đích |
| `p_history_limit` | `1000` | Số commit tối đa đọc khi xét 1 bảng đầu vào; vượt → coi cạnh là bẩn |
| `p_non_data_ops` | `""` | Ghi đè danh sách operation không đổi dữ liệu (CSV); trống = `DEFAULT_NON_DATA_OPS` của lib |

Kết quả: `ctrl_log_run` (1 dòng / lần chạy), `ctrl_log_table_run` (1 dòng / node), cạnh trong `ctrl_mng_watermark`.
Node lỗi → run `PARTIAL_FAILED` và notebook raise (pipeline báo lỗi); lần sau chỉ chạy lại phần còn bẩn.

Bảng control ở `lh_vv_ctrl.dbo` (hằng `CTRL_SCHEMA` của `NB_LIB_TRANSFORM_SLV_GLD`).

Sau cutover 08/10, notebook raise `ConfigError` khi `p_pl_name` chưa có cạnh `RECOMPUTE` trong `lh_vv_ctrl`. Lỗi này xảy ra trước khi tạo khoá `wm_flow__<pl>` và trước khi ghi log. Không có cạnh thì không chạy node và không ghi bảng gold.


In [ ]:
p_pl_name = "PL_VV_TRANSFORM_SLV_TO_GLD_1H"          # bắt buộc, pipeline truyền (chạy tay: điền tên pl)
p_run_id = ""
p_mode = "RUN"
p_max_parallel = "3"
p_node_timeout_min = "20"
p_dag_timeout_min = "45"
p_lock_timeout_min = "90"
p_force_nodes = ""
p_history_limit = "1000"
p_non_data_ops = ""

In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD

In [ ]:
RUN_SUMMARY = orchestrate(
    p_pl_name=p_pl_name, p_run_id=p_run_id, p_mode=p_mode, p_max_parallel=p_max_parallel,
    p_node_timeout_min=p_node_timeout_min, p_dag_timeout_min=p_dag_timeout_min, p_lock_timeout_min=p_lock_timeout_min,
    p_force_nodes=p_force_nodes, p_history_limit=p_history_limit, p_non_data_ops=p_non_data_ops)
print(json_dumps(RUN_SUMMARY))